# Cutout test notebook

Test the vo-cutouts service by requesting a cutout of a static dataset ID for an image in DP1.
This avoids requiring other services, such as TAP or DataLink, at the cost of not matching the normal flow that a user would use with PyVO.

## Imports

In [ ]:
from lsst.afw import display as afwDisplay
from lsst.rsp import RSPDiscovery
from matplotlib import pyplot
from lsst.images.serialization import read as read_archive

## Client setup

Create an authenticated client and get the URL of the cutout service.

In [ ]:
discovery = RSPDiscovery("dp1")
session = discovery.get_session()
url = discovery.get_service_url("cutout", version="soda-sync-1.0")
print(f"Base sync cutout URL: {url}")

## Cutout parameters

This hard-coded image ID is from the 103_4_Small_Image_Cutout tutorial for DP1.

In [ ]:
image_id = "ivo://org.rubinobs/usdac/lsst-dp2?repo=dp2&id=019ed8bf-a01f-7fd7-bc6b-8af89d2e0be4"
ra = 53.1246023
dec = -27.7404715
radius = 0.01

## Sync cutout request

Make a sync request to the image cutout service and store the resulting cutout in memory.
Request the full exposure to allow use of afw for the display.

In [ ]:
r = session.get(url, params={"id": image_id, "cutoutdetail": "Exposure", "circle": f"{ra} {dec} {radius}"}, timeout=60)
r.raise_for_status()

## Read results

Create an exposure from that image.

In [ ]:
import tempfile
with tempfile.NamedTemporaryFile(suffix=".fits") as tmp:
    tmp.write(r.content)
    tmp.seek(0)
    cutout = read_archive(tmp.name)
print(cutout)

## Display the cutout

Display the resulting cutout.

In [ ]:
afwDisplay.setDefaultBackend("matplotlib")
display = afwDisplay.Display()
display.scale('asinh', 'zscale')
# This seems to be broken at the moment.
# display.image(cutout.to_legacy().image)
# pyplot.show()